# 🧪 Testing and Debugging

**Unified Military Analytics and Comparison Dashboard — Module 7**

This notebook validates the data pipeline and the Streamlit dashboard's core logic *before* it is exercised manually in the browser. It covers:

1. Data Loading & Integrity
2. Filter Options & Cascading Filters
3. KPI Calculations
4. Country Profile & Long-Format Metrics Lookup
5. Quick Stats Chart Rendering (`charts.py`)
6. Coalition Builder Aggregation Logic
7. Edge Case & Empty-State Handling
8. QA Summary & Export to `QA_Test_Results.xlsx`



In [1]:
import os
import sys

def find_dashboard_dir(start_path, max_depth=4):
    """Walks up and down from start_path looking for a folder that
    contains BOTH utils.py and charts.py -- the actual dashboard code."""
    start_path = os.path.abspath(start_path)

    # Search upward first (in case notebook cwd is deep inside the repo)
    current = start_path
    for _ in range(max_depth):
        for root, dirs, files in os.walk(current):
            depth = root[len(current):].count(os.sep)
            if depth > max_depth:
                dirs[:] = []
                continue
            if "utils.py" in files and "charts.py" in files:
                return root
        current = os.path.dirname(current)

    return None

DASHBOARD_DIR = find_dashboard_dir(os.getcwd())

if DASHBOARD_DIR is None:
    raise FileNotFoundError(
        "Could not locate a folder containing both utils.py and charts.py "
        f"searching from {os.getcwd()!r}. Run the Step 1 listing cell above "
        "to find the correct folder name, then set DASHBOARD_DIR manually, e.g.:\n"
        '    DASHBOARD_DIR = "/Users/mac/Downloads/Unified-Military-Analytics/<exact folder name>"'
    )

if DASHBOARD_DIR not in sys.path:
    sys.path.insert(0, DASHBOARD_DIR)

import utils
import charts

print("Dashboard modules loaded from:", DASHBOARD_DIR)

2026-08-22 22:29:05.630 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


Dashboard modules loaded from: /Users/mac/Desktop/Unified-Military-Analytics/Milestone 3: Full Dashboard Development


In [2]:
import os

print("cwd:", os.getcwd())
print()
print("Contents of cwd:")
for item in os.listdir(os.getcwd()):
    print(" -", item)

# If you can see your project folder from here, list inside it too:
project_guess = os.path.join(os.getcwd(), "Unified-Military-Analytics")
if os.path.exists(project_guess):
    print()
    print("Contents of Unified-Military-Analytics:")
    for item in os.listdir(project_guess):
        print(" -", item)

cwd: /Users/mac/Desktop/Unified-Military-Analytics/Module 7: Testing and Debugging

Contents of cwd:
 - Testing_and_Debugging.ipynb
 - QA_Test_Results.xlsx
 - QA checklist.md


## 0. Setup & Imports

In [3]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)

# ---------------------------------------------------------------------------
# Locate and import the dashboard's own modules (utils.py, charts.py) so we
# are testing the EXACT code the live app runs, not a copy/paste of it.
# This notebook lives in "Module7-Testing and Debugging/", and the app lives
# one level up in "Module5&6-Full Dashboard Development/".
# ---------------------------------------------------------------------------
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DASHBOARD_DIR = os.path.join(PROJECT_ROOT, "Module5&6-Full Dashboard Development")

if DASHBOARD_DIR not in sys.path:
    sys.path.insert(0, DASHBOARD_DIR)

import utils
import charts

print("Dashboard modules loaded from:", DASHBOARD_DIR)
print("utils.py  ->", utils.__file__)
print("charts.py ->", charts.__file__)

Dashboard modules loaded from: /Users/mac/Desktop/Unified-Military-Analytics/Module5&6-Full Dashboard Development
utils.py  -> /Users/mac/Desktop/Unified-Military-Analytics/Milestone 3: Full Dashboard Development/utils.py
charts.py -> /Users/mac/Desktop/Unified-Military-Analytics/Milestone 3: Full Dashboard Development/charts.py


In [4]:
# ---------------------------------------------------------------------------
# QA TEST HARNESS
# ---------------------------------------------------------------------------
qa_results = []

def run_test(test_id, module, description, func):
    """Runs `func`, catches any assertion/exception, and logs the outcome."""
    try:
        func()
        status, note = "PASS", ""
    except AssertionError as e:
        status, note = "FAIL", str(e)
    except Exception as e:
        status, note = "ERROR", f"{type(e).__name__}: {e}"

    qa_results.append({
        "Test ID": test_id,
        "Module": module,
        "Description": description,
        "Status": status,
        "Notes": note,
    })
    flag = {"PASS": "✅", "FAIL": "❌", "ERROR": "🛑"}[status]
    print(f"{flag} [{status}] {test_id} — {description}" + (f"  |  {note}" if note else ""))

## 1. Data Loading & Integrity

Confirms `utils.load_data()` returns the two expected datasets, with the coverage and cleanliness required by the project spec (**≥ 140 countries, < 2% missing data, no structural errors**).

In [5]:
final_df, long_df = utils.load_data()

REQUIRED_COLUMNS = [
    utils.COL_COUNTRY, utils.COL_RANK, utils.COL_SCORE, utils.COL_BUDGET,
    utils.COL_ACTIVE, utils.COL_RESERVE, utils.COL_AIRCRAFT, utils.COL_TANKS,
    utils.COL_NAVAL, utils.COL_GDP, utils.COL_POP,
    utils.COL_CONTINENT, utils.COL_REGION, utils.COL_ALLIANCE,
]


def test_data_loaded():
    assert final_df is not None and not final_df.empty, "final_df failed to load or is empty"
    assert long_df is not None and not long_df.empty, "long_df failed to load or is empty"

run_test("T1.1", "Data Loading", "military_final.xlsx and military_long.xlsx both load successfully", test_data_loaded)


def test_country_coverage():
    n = final_df[utils.COL_COUNTRY].nunique()
    assert n >= 140, f"Expected >= 140 countries, found {n}"

run_test("T1.2", "Data Loading", "Dataset covers at least 140 countries", test_country_coverage)


def test_required_columns():
    missing = [c for c in REQUIRED_COLUMNS if c not in final_df.columns]
    assert not missing, f"Missing required column(s): {missing}"

run_test("T1.3", "Data Loading", "All required KPI/profile columns are present", test_required_columns)


def test_missing_value_threshold():
    core_cols = [c for c in REQUIRED_COLUMNS if c in final_df.columns]
    missing_pct = final_df[core_cols].isna().mean().mean() * 100
    assert missing_pct < 2.0, f"Missing data is {missing_pct:.2f}% (target < 2%)"

run_test("T1.4", "Data Loading", "Missing data across core columns is under 2%", test_missing_value_threshold)


def test_no_duplicate_countries():
    dupes = final_df[utils.COL_COUNTRY].duplicated().sum()
    assert dupes == 0, f"Found {dupes} duplicate country row(s)"

run_test("T1.5", "Data Loading", "No duplicate country rows in the dataset", test_no_duplicate_countries)


def test_rank_is_unique_and_positive():
    ranks = final_df[utils.COL_RANK].dropna()
    assert (ranks > 0).all(), "Found non-positive Power Index Rank value(s)"
    assert ranks.duplicated().sum() == 0, "Power Index Rank has duplicate values"

run_test("T1.6", "Data Loading", "Power Index Rank values are unique and positive", test_rank_is_unique_and_positive)

2026-08-22 22:29:06.259 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-08-22 22:29:06.262 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


✅ [PASS] T1.1 — military_final.xlsx and military_long.xlsx both load successfully
✅ [PASS] T1.2 — Dataset covers at least 140 countries
✅ [PASS] T1.3 — All required KPI/profile columns are present
✅ [PASS] T1.4 — Missing data across core columns is under 2%
✅ [PASS] T1.5 — No duplicate country rows in the dataset
✅ [PASS] T1.6 — Power Index Rank values are unique and positive


## 2. Filter Options & Cascading Filters

Confirms filters behave the way `app.py`'s sidebar expects: an empty selection returns the full dataset, a single filter narrows correctly, and combining filters narrows further (never impossibly, never inconsistently).

In [6]:
def test_no_filters_returns_full_dataset():
    out = utils.apply_filters(final_df, countries=[], regions=[], continents=[], alliances=[])
    assert len(out) == len(final_df), "Empty filter selection should return the full dataset"

run_test("T2.1", "Filters", "apply_filters() with no selections returns the full, unfiltered dataset", test_no_filters_returns_full_dataset)


def test_single_country_filter():
    sample_country = final_df[utils.COL_COUNTRY].iloc[0]
    out = utils.apply_filters(final_df, countries=[sample_country], regions=[], continents=[], alliances=[])
    assert len(out) == 1, f"Expected exactly 1 row for {sample_country!r}, got {len(out)}"
    assert out[utils.COL_COUNTRY].iloc[0] == sample_country

run_test("T2.2", "Filters", "Filtering by a single country returns exactly that one row", test_single_country_filter)


def test_continent_filter_matches_manual_subset():
    sample_continent = final_df[utils.COL_CONTINENT].dropna().iloc[0]
    out = utils.apply_filters(final_df, countries=[], regions=[], continents=[sample_continent], alliances=[])
    manual = final_df[final_df[utils.COL_CONTINENT] == sample_continent]
    assert len(out) == len(manual), "Continent filter result does not match a manual pandas filter"

run_test("T2.3", "Filters", "Continent filter output matches an equivalent manual pandas filter", test_continent_filter_matches_manual_subset)


def test_cascading_options_are_consistent():
    """
    Mirrors the cascading-filter logic used in app.py's sidebar: when a
    country is selected, that country's OWN continent/region should still
    appear in the option lists computed for the Continent/Region filters
    (i.e. selecting a country never orphans its own metadata values).
    """
    sample_country = final_df[utils.COL_COUNTRY].iloc[0]
    row = final_df[final_df[utils.COL_COUNTRY] == sample_country].iloc[0]
    true_continent = row[utils.COL_CONTINENT]
    true_region = row[utils.COL_REGION]

    subset_for_continent = utils.apply_filters(final_df, countries=[sample_country], regions=[], continents=[], alliances=[])
    subset_for_region = utils.apply_filters(final_df, countries=[sample_country], regions=[], continents=[], alliances=[])

    continent_options = sorted(subset_for_continent[utils.COL_CONTINENT].dropna().unique().tolist())
    region_options = sorted(subset_for_region[utils.COL_REGION].dropna().unique().tolist())

    assert true_continent in continent_options, f"{true_continent!r} missing from cascaded Continent options for {sample_country!r}"
    assert true_region in region_options, f"{true_region!r} missing from cascaded Region options for {sample_country!r}"

run_test("T2.4", "Filters", "Cascading filter options remain consistent with the selected country's own metadata", test_cascading_options_are_consistent)


def test_impossible_combination_returns_empty_not_error():
    sample_country = final_df[utils.COL_COUNTRY].iloc[0]
    true_continent = final_df[final_df[utils.COL_COUNTRY] == sample_country][utils.COL_CONTINENT].iloc[0]
    other_continents = [c for c in final_df[utils.COL_CONTINENT].dropna().unique() if c != true_continent]
    assert other_continents, "Dataset only has one continent value -- cannot test this case"

    out = utils.apply_filters(final_df, countries=[sample_country], regions=[], continents=[other_continents[0]], alliances=[])
    assert len(out) == 0, "An impossible country/continent combination should return zero rows, not raise or return stale data"

run_test("T2.5", "Filters", "A contradictory country + continent combination returns an empty result without raising", test_impossible_combination_returns_empty_not_error)

✅ [PASS] T2.1 — apply_filters() with no selections returns the full, unfiltered dataset
✅ [PASS] T2.2 — Filtering by a single country returns exactly that one row
✅ [PASS] T2.3 — Continent filter output matches an equivalent manual pandas filter
✅ [PASS] T2.4 — Cascading filter options remain consistent with the selected country's own metadata
✅ [PASS] T2.5 — A contradictory country + continent combination returns an empty result without raising


## 3. KPI Calculations

Confirms `utils.compute_kpis()` produces sane values on both the full dataset and a filtered subset.

In [7]:
def test_kpis_on_full_dataset():
    kpis = utils.compute_kpis(final_df)
    assert kpis["total_countries"] == final_df[utils.COL_COUNTRY].nunique()
    assert kpis["best_country"], "best_country KPI is empty"

run_test("T3.1", "KPIs", "compute_kpis() on the full dataset returns internally consistent totals", test_kpis_on_full_dataset)


def test_kpis_on_filtered_subset():
    sample_alliance = final_df[utils.COL_ALLIANCE].dropna().iloc[0]
    subset = utils.apply_filters(final_df, countries=[], regions=[], continents=[], alliances=[sample_alliance])
    kpis = utils.compute_kpis(subset)
    assert kpis["total_countries"] == subset[utils.COL_COUNTRY].nunique()
    assert kpis["total_countries"] <= final_df[utils.COL_COUNTRY].nunique()

run_test("T3.2", "KPIs", "compute_kpis() on a filtered subset never exceeds the full-dataset totals", test_kpis_on_filtered_subset)


def test_best_country_is_top_ranked():
    kpis = utils.compute_kpis(final_df)
    top_ranked = final_df.loc[final_df[utils.COL_RANK].idxmin(), utils.COL_COUNTRY]
    assert kpis["best_country"] == top_ranked, (
        f"best_country KPI ({kpis['best_country']!r}) does not match the #1 ranked country ({top_ranked!r})"
    )

run_test("T3.3", "KPIs", "best_country KPI matches the country with the lowest (strongest) Power Index Rank", test_best_country_is_top_ranked)

✅ [PASS] T3.1 — compute_kpis() on the full dataset returns internally consistent totals
✅ [PASS] T3.2 — compute_kpis() on a filtered subset never exceeds the full-dataset totals
✅ [PASS] T3.3 — best_country KPI matches the country with the lowest (strongest) Power Index Rank


## 4. Country Profile & Long-Format Metrics Lookup

Confirms the Nation Overview / Country Search data-access functions behave correctly for both a valid country and an unknown one.

In [8]:
def test_valid_country_profile():
    sample_country = final_df[utils.COL_COUNTRY].iloc[0]
    profile = utils.get_country_profile(final_df, sample_country)
    assert profile, f"No profile returned for a known country: {sample_country!r}"

run_test("T4.1", "Country Profile", "get_country_profile() returns data for a known, valid country", test_valid_country_profile)


def test_unknown_country_profile_handled_gracefully():
    profile = utils.get_country_profile(final_df, "Wakanda (Not A Real Country)")
    assert not profile, "An unknown country should return an empty/falsy profile, not raise or fabricate data"

run_test("T4.2", "Country Profile", "get_country_profile() handles an unknown country name gracefully", test_unknown_country_profile_handled_gracefully)


def test_long_metrics_lookup():
    sample_country = final_df[utils.COL_COUNTRY].iloc[0]
    metrics = utils.get_country_long_metrics(long_df, sample_country)
    assert metrics is not None, "get_country_long_metrics() returned None for a known country"

run_test("T4.3", "Country Profile", "get_country_long_metrics() returns raw metric rows for a known country", test_long_metrics_lookup)

✅ [PASS] T4.1 — get_country_profile() returns data for a known, valid country
✅ [PASS] T4.2 — get_country_profile() handles an unknown country name gracefully
✅ [PASS] T4.3 — get_country_long_metrics() returns raw metric rows for a known country


## 5. Quick Stats Chart Rendering (`charts.py`)

Every chart-building function in `charts.py` is called against the **full dataset** and against a **deliberately empty dataset** — the latter must return a clean "no data" figure instead of raising, exactly as `charts._no_data_figure()` is designed to do.

In [9]:
empty_df = final_df.iloc[0:0].copy()

CHART_FUNCTIONS = [
    ("top_power_index_chart", charts.top_power_index_chart),
    ("top_defense_budget_chart", charts.top_defense_budget_chart),
    ("top_military_assets_chart", charts.top_military_assets_chart),
    ("top_ppp_chart", charts.top_ppp_chart),
    ("top_aircraft_chart", charts.top_aircraft_chart),
    ("top_tanks_chart", charts.top_tanks_chart),
    ("top_naval_chart", charts.top_naval_chart),
    ("defense_budget_bubble_chart", charts.defense_budget_bubble_chart),
    ("continent_distribution_chart", charts.continent_distribution_chart),
    ("alliance_distribution_chart", charts.alliance_distribution_chart),
    ("military_capability_comparison_chart", charts.military_capability_comparison_chart),
]

for name, fn in CHART_FUNCTIONS:

    def _test(fn=fn, name=name):
        fig = fn(final_df)
        assert isinstance(fig, go.Figure), f"{name}() did not return a Plotly Figure on the full dataset"

    run_test(f"T5.{name}", "Charts (Quick Stats)", f"{name}() renders a valid Figure on the full dataset", _test)

for name, fn in CHART_FUNCTIONS:

    def _test_empty(fn=fn, name=name):
        fig = fn(empty_df)
        assert isinstance(fig, go.Figure), f"{name}() did not return a Plotly Figure on an empty dataset"

    run_test(f"T5.{name}.empty", "Charts (Quick Stats)", f"{name}() falls back to a clean 'no data' Figure on an empty dataset", _test_empty)

✅ [PASS] T5.top_power_index_chart — top_power_index_chart() renders a valid Figure on the full dataset
✅ [PASS] T5.top_defense_budget_chart — top_defense_budget_chart() renders a valid Figure on the full dataset
✅ [PASS] T5.top_military_assets_chart — top_military_assets_chart() renders a valid Figure on the full dataset
✅ [PASS] T5.top_ppp_chart — top_ppp_chart() renders a valid Figure on the full dataset
✅ [PASS] T5.top_aircraft_chart — top_aircraft_chart() renders a valid Figure on the full dataset
✅ [PASS] T5.top_tanks_chart — top_tanks_chart() renders a valid Figure on the full dataset
✅ [PASS] T5.top_naval_chart — top_naval_chart() renders a valid Figure on the full dataset
✅ [PASS] T5.defense_budget_bubble_chart — defense_budget_bubble_chart() renders a valid Figure on the full dataset
✅ [PASS] T5.continent_distribution_chart — continent_distribution_chart() renders a valid Figure on the full dataset
✅ [PASS] T5.alliance_distribution_chart — alliance_distribution_chart() renders

In [10]:
def test_military_rank_map_full():
    fig, is_choropleth = charts.military_rank_map(final_df)
    assert isinstance(fig, go.Figure), "military_rank_map() did not return a Plotly Figure"
    assert isinstance(is_choropleth, bool), "military_rank_map() second return value should be a bool"

run_test("T5.map.full", "Charts (Quick Stats)", "military_rank_map() renders on the full dataset (choropleth or bar fallback)", test_military_rank_map_full)


def test_military_rank_map_empty():
    fig, is_choropleth = charts.military_rank_map(empty_df)
    assert isinstance(fig, go.Figure), "military_rank_map() did not return a Plotly Figure on empty data"
    assert is_choropleth is False, "An empty dataset should never be reported as a successful choropleth render"

run_test("T5.map.empty", "Charts (Quick Stats)", "military_rank_map() falls back cleanly on an empty dataset", test_military_rank_map_empty)

✅ [PASS] T5.map.full — military_rank_map() renders on the full dataset (choropleth or bar fallback)
✅ [PASS] T5.map.empty — military_rank_map() falls back cleanly on an empty dataset


## 6. Coalition Builder Aggregation Logic

`pages/coalition_builder.py` is a Streamlit script (it calls `st.*` and `st.stop()` at import time), so it cannot be safely `import`-ed outside a running Streamlit session. The pure calculation logic is mirrored below — copied verbatim from `coalition_builder.py` — so it can be unit-tested in isolation, exactly as it behaves in the live app.

In [11]:
# --- Mirrored verbatim from pages/coalition_builder.py, for isolated testing ---

RANK_COL = utils.COL_RANK
SCORE_COL = utils.COL_SCORE
COUNTRY_COL = utils.COL_COUNTRY

KPI_METRICS_COLS = [
    "total_military_manpower", "active_personnel", "reserve_personnel",
    "total_military_aircraft", "fighter_aircraft", "tanks",
    "total_naval_fleet", "submarines", "defense_budget_usd", "GDP",
    "total_population",
]


def pct_delta(value, base):
    if base is None or pd.isna(base) or base == 0:
        return None
    return (value - base) / base * 100.0


def aggregate_group(group_df: pd.DataFrame) -> dict:
    agg = {"count": int(group_df.shape[0])}

    numeric_cols = set(group_df.select_dtypes(include=[np.number]).columns)
    for col in set(KPI_METRICS_COLS):
        if col in numeric_cols:
            agg[col] = float(group_df[col].sum())
        else:
            agg[col] = 0.0

    if group_df.empty:
        agg["avg_power_index_score"] = None
        agg["best_rank"] = None
        agg["best_rank_country"] = None
        agg["agg_budget_to_gdp"] = None
    else:
        agg["avg_power_index_score"] = float(group_df[SCORE_COL].mean())
        best_row = group_df.loc[group_df[RANK_COL].idxmin()]
        agg["best_rank"] = int(best_row[RANK_COL])
        agg["best_rank_country"] = best_row[COUNTRY_COL]
        total_gdp = agg.get("GDP", 0.0)
        agg["agg_budget_to_gdp"] = (agg.get("defense_budget_usd", 0.0) / total_gdp * 100.0) if total_gdp else None

    return agg

In [12]:
def test_manpower_is_summed_across_group():
    sample = final_df.dropna(subset=["total_military_manpower"]).head(3)
    agg = aggregate_group(sample)
    expected = float(sample["total_military_manpower"].sum())
    assert abs(agg["total_military_manpower"] - expected) < 1e-6, "Manpower should be SUMMED across a coalition, not averaged"

run_test("T6.1", "Coalition Builder", "Additive metrics (e.g. manpower) are summed across coalition members", test_manpower_is_summed_across_group)


def test_power_index_is_averaged_not_summed():
    sample = final_df.dropna(subset=[SCORE_COL]).head(3)
    agg = aggregate_group(sample)
    expected_avg = float(sample[SCORE_COL].mean())
    expected_sum = float(sample[SCORE_COL].sum())
    assert abs(agg["avg_power_index_score"] - expected_avg) < 1e-9, "avg_power_index_score does not match a plain mean()"
    assert abs(agg["avg_power_index_score"] - expected_sum) > 1e-6, "Power Index Score must be AVERAGED, never summed"

run_test("T6.2", "Coalition Builder", "Power Index Score is averaged (never summed) across coalition members", test_power_index_is_averaged_not_summed)


def test_best_rank_member_is_correct():
    sample = final_df.dropna(subset=[RANK_COL]).head(5)
    agg = aggregate_group(sample)
    true_best = sample.loc[sample[RANK_COL].idxmin(), COUNTRY_COL]
    assert agg["best_rank_country"] == true_best, "best_rank_country does not match the lowest Power Index Rank in the group"

run_test("T6.3", "Coalition Builder", "The 'best ranked member' of a coalition is the country with the lowest Power Index Rank", test_best_rank_member_is_correct)


def test_budget_to_gdp_is_group_level_ratio_not_averaged_per_country():
    sample = final_df.dropna(subset=["defense_budget_usd", "GDP"]).head(3)
    agg = aggregate_group(sample)
    expected_ratio = agg["defense_budget_usd"] / agg["GDP"] * 100.0
    assert abs(agg["agg_budget_to_gdp"] - expected_ratio) < 1e-6, (
        "Budget-to-GDP should be (total budget / total GDP), not an average of each country's own ratio"
    )

run_test("T6.4", "Coalition Builder", "Budget-to-GDP ratio is computed at the group level (total / total), not averaged per-country", test_budget_to_gdp_is_group_level_ratio_not_averaged_per_country)


def test_empty_coalition_handled_safely():
    empty_group = final_df.iloc[0:0]
    agg = aggregate_group(empty_group)
    assert agg["count"] == 0
    assert agg["avg_power_index_score"] is None, "Empty coalition should report None, not 0 or NaN, for average Power Index Score"
    assert agg["best_rank_country"] is None

run_test("T6.5", "Coalition Builder", "An empty coalition group is handled safely (no crash, no misleading zero)", test_empty_coalition_handled_safely)


def test_pct_delta_handles_zero_base():
    assert pct_delta(100.0, 0) is None, "pct_delta() should return None (not raise ZeroDivisionError) when the reference value is 0"
    assert pct_delta(100.0, 50.0) == 100.0

run_test("T6.6", "Coalition Builder", "pct_delta() safely returns None instead of dividing by zero", test_pct_delta_handles_zero_base)

✅ [PASS] T6.1 — Additive metrics (e.g. manpower) are summed across coalition members
✅ [PASS] T6.2 — Power Index Score is averaged (never summed) across coalition members
✅ [PASS] T6.3 — The 'best ranked member' of a coalition is the country with the lowest Power Index Rank
✅ [PASS] T6.4 — Budget-to-GDP ratio is computed at the group level (total / total), not averaged per-country
✅ [PASS] T6.5 — An empty coalition group is handled safely (no crash, no misleading zero)
✅ [PASS] T6.6 — pct_delta() safely returns None instead of dividing by zero


## 7. Edge Case & Empty-State Handling

Confirms the dashboard's defensive checks (used throughout `app.py` / the page modules) behave as designed when the user's selections produce no data.

In [13]:
def test_fully_filtered_out_selection_is_empty_not_error():
    out = utils.apply_filters(final_df, countries=["Definitely Not A Real Country"], regions=[], continents=[], alliances=[])
    assert isinstance(out, pd.DataFrame)
    assert out.empty, "Filtering by a nonexistent country should return an empty DataFrame, not raise"

run_test("T7.1", "Edge Cases", "Filtering by a nonexistent country returns an empty DataFrame instead of raising", test_fully_filtered_out_selection_is_empty_not_error)


def test_charts_never_raise_on_all_nan_column():
    broken_df = final_df.copy()
    broken_df["defense_budget_usd"] = np.nan
    fig = charts.top_defense_budget_chart(broken_df)
    assert isinstance(fig, go.Figure), "A fully-NaN metric column should fall back to a clean 'no data' figure, not raise"

run_test("T7.2", "Edge Cases", "An all-NaN metric column does not crash chart rendering", test_charts_never_raise_on_all_nan_column)


def test_bubble_chart_handles_missing_optional_columns():
    stripped_df = final_df.drop(columns=["tanks", "total_naval_fleet"], errors="ignore")
    fig = charts.defense_budget_bubble_chart(stripped_df)
    assert isinstance(fig, go.Figure), "Bubble chart should degrade gracefully when optional hover columns are missing"

run_test("T7.3", "Edge Cases", "Defense budget bubble chart degrades gracefully when optional columns are absent", test_bubble_chart_handles_missing_optional_columns)


def test_single_country_group_budget_ratio_with_zero_gdp():
    fake_row = final_df.iloc[[0]].copy()
    fake_row["GDP"] = 0.0
    agg = aggregate_group(fake_row)
    assert agg["agg_budget_to_gdp"] is None, "A zero-GDP coalition should report None for Budget-to-GDP, not raise or divide by zero"

run_test("T7.4", "Edge Cases", "Coalition aggregation handles a zero-GDP group without raising", test_single_country_group_budget_ratio_with_zero_gdp)

✅ [PASS] T7.1 — Filtering by a nonexistent country returns an empty DataFrame instead of raising
✅ [PASS] T7.2 — An all-NaN metric column does not crash chart rendering
✅ [PASS] T7.3 — Defense budget bubble chart degrades gracefully when optional columns are absent
✅ [PASS] T7.4 — Coalition aggregation handles a zero-GDP group without raising


## 8. QA Summary & Export

Collates every test above into a single QA log and exports it to `QA_Test_Results.xlsx`, matching the Module 7 deliverable list in the project README.

In [14]:
qa_df = pd.DataFrame(qa_results)

summary = qa_df["Status"].value_counts().reindex(["PASS", "FAIL", "ERROR"], fill_value=0)
total = len(qa_df)

print("=" * 60)
print("QA SUMMARY")
print("=" * 60)
for status, count in summary.items():
    print(f"{status:>6}: {count} / {total}")
print("=" * 60)

qa_df

QA SUMMARY
  PASS: 51 / 51
  FAIL: 0 / 51
 ERROR: 0 / 51


,Test ID,Module,Description,Status,Notes
0,T1.1,Data Loading,military_final.xlsx and military_long.xlsx bot...,PASS,
1,T1.2,Data Loading,Dataset covers at least 140 countries,PASS,
2,T1.3,Data Loading,All required KPI/profile columns are present,PASS,
3,T1.4,Data Loading,Missing data across core columns is under 2%,PASS,
4,T1.5,Data Loading,No duplicate country rows in the dataset,PASS,
5,T1.6,Data Loading,Power Index Rank values are unique and positive,PASS,
6,T2.1,Filters,apply_filters() with no selections returns the...,PASS,
7,T2.2,Filters,Filtering by a single country returns exactly ...,PASS,
8,T2.3,Filters,Continent filter output matches an equivalent ...,PASS,
9,T2.4,Filters,Cascading filter options remain consistent wit...,PASS,


In [15]:
OUTPUT_PATH = os.path.join(os.getcwd(), "QA_Test_Results.xlsx")
qa_df.to_excel(OUTPUT_PATH, index=False, sheet_name="QA Results")
print(f"QA results exported to: {OUTPUT_PATH}")

QA results exported to: /Users/mac/Desktop/Unified-Military-Analytics/Module 7: Testing and Debugging/QA_Test_Results.xlsx
